# ETL

## Enviroment setup

To keep constant track of the files used in this Notebook, I will use my own Google Drive to store the datasets and models.

In [1]:
# Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In addition, to enable other people to run this Notebook, I defined a dictionary of paths to maintain consistency throughout the Notebook.

In [2]:
# Paths

BASE_PATH = "/content/drive/MyDrive/TEC/8vo IA/Proyecto IA"

PATHS = {
  "RAW_DATA_PATH": f"{BASE_PATH}/data/raw",
  "CLEAN_DATA_PATH": f"{BASE_PATH}/data/clean",
}

In [3]:
# Data manipulation
import pandas as pd
import numpy as np

# Natural Language Toolkit
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

# SciKit Learn
from sklearn.model_selection import train_test_split

# Kaggle
import kagglehub

# Utilities
import os
import json
from datetime import datetime, timedelta
from pathlib import Path

# PyArrow for efficient Parquet writing
import pyarrow as pa
import pyarrow.parquet as pq

## Data collection & exploration

The Yelp Academic Dataset comprises 6,990,280 reviews distributed across multiple JSON files, totaling 8.65 GB. For this analysis, I focus exclusively on the yelp_academic_dataset_review.json file (5.09 GB), which contains the core review data including text content, star ratings, timestamps, and user engagement metrics.

This file provides all necessary features for sentiment analysis, eliminating the need for additional business or user metadata files.

In [4]:
#@title YelpDataLoader

class YelpDataLoader:
  def __init__(self):
    self.KAGGLE_ID   = "yelp-dataset/yelp-dataset"
    self.OUTPUT_DIR  = Path(PATHS["RAW_DATA_PATH"])
    self.OUTPUT_FILE = self.OUTPUT_DIR / "yelp_reviews_raw.parquet"
    self.kaggle_path = None

  def extract(self, chunk_size: int = 100_000) -> pd.DataFrame:
    """
    Download the Yelp dataset from Kaggle and read the reviews JSON.
    Returns a DataFrame with all reviews.
    """
    print("Running Kaggle extraction…")
    print(f"  Dataset : {self.KAGGLE_ID}")

    self.kaggle_path = kagglehub.dataset_download(self.KAGGLE_ID)
    print(f"  Path    : {self.kaggle_path}")

    reviews_file = os.path.join(self.kaggle_path, "yelp_academic_dataset_review.json")
    print(f"  File    : {reviews_file}")
    print(f"  Chunk size: {chunk_size:,} records\n")

    chunks = []
    for i, chunk in enumerate(pd.read_json(reviews_file, lines=True, chunksize=chunk_size), start=1):
      print(f"  Chunk {i}: {len(chunk):,} records")
      chunks.append(chunk)

    df = pd.concat(chunks, ignore_index=True)
    print(f"\n  Total rows loaded: {len(df):,}")
    return df

  def file_exists(self, path: Path = None) -> bool:
    target = path or self.OUTPUT_FILE
    return target.exists()

  def save_raw(self, df: pd.DataFrame, path: Path = None) -> Path:
    target = path or self.OUTPUT_FILE
    target.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(target, index=False)
    print(f"  Raw data saved → {target}  ({target.stat().st_size / 1e6:.1f} MB)")
    return target

  def load(self, force_download: bool = False, chunk_size: int = 100_000) -> pd.DataFrame:
    """
    Return cached parquet if it exists, otherwise download and cache it.

      df = YelpDataLoader().load()
    """
    if not force_download and self.file_exists():
      print(f"  Cache hit — loading from {self.OUTPUT_FILE}")
      return pd.read_parquet(self.OUTPUT_FILE)

    df = self.extract(chunk_size=chunk_size)
    self.save_raw(df)
    return df

  def sample(
    self,
    df: pd.DataFrame,
    output_path: Path,
    fraction: float = 0.2,
    seed: int = 42,
    force: bool = False,
  ) -> pd.DataFrame:
    """
    Return a reproducible stratified sample, cached as parquet.

      df_sample = loader.sample(df, Path("data/yelp_sample.parquet"))
    """
    output_path = Path(output_path)

    if not force and self.file_exists(output_path):
      print(f"  Cache hit — loading sample from {output_path}")
      return pd.read_parquet(output_path)

    print(f"  Sampling {fraction * 100:.0f}% of {len(df):,} reviews…")
    df_sample = df.sample(frac=fraction, random_state=seed)
    print(f"  Sample size: {len(df_sample):,} reviews")

    self.save_raw(df_sample, output_path)
    return df_sample

  def explore_kaggle_dataset(self) -> None:
    if self.kaggle_path is None:
      print("  Dataset was loaded from cache — no Kaggle path available.")
      return

    print(f"\nContents of: {self.kaggle_path}\n")
    total_size = 0

    for root, _, files in os.walk(self.kaggle_path):
      for file in files:
        file_path = os.path.join(root, file)
        size = os.path.getsize(file_path)
        total_size += size
        print(f"  {file:40s}  {size / 1e6:8.2f} MB")

    print(f"\n  Total: {total_size / 1e9:.2f} GB ({total_size / 1e6:.0f} MB)")

  def explore_reviews_dataset(self, df: pd.DataFrame) -> None:
    print("\n── Yelp Reviews Info ──────────────────────────")
    print(df.dtypes)
    print()
    df.info()

    print(f"\n  Total reviews : {len(df):,}")

    print("\n── Sample rows ────────────────────────────────")
    print(df.head(5).to_string())

    print("\n── Stars distribution ─────────────────────────")
    print(df["stars"].value_counts().sort_index())

### Load dataset

In [5]:
loader = YelpDataLoader()

df = loader.load()

  Cache hit — loading from /content/drive/MyDrive/TEC/8vo IA/Proyecto IA/data/raw/yelp_reviews_raw.parquet


In [6]:
loader.explore_kaggle_dataset()

  Dataset was loaded from cache — no Kaggle path available.


In [7]:
loader.explore_reviews_dataset(df)


── Yelp Reviews Info ──────────────────────────
review_id              object
user_id                object
business_id            object
stars                   int64
useful                  int64
funny                   int64
cool                    int64
text                   object
date           datetime64[ns]
dtype: object

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6990280 entries, 0 to 6990279
Data columns (total 9 columns):
 #   Column       Dtype         
---  ------       -----         
 0   review_id    object        
 1   user_id      object        
 2   business_id  object        
 3   stars        int64         
 4   useful       int64         
 5   funny        int64         
 6   cool         int64         
 7   text         object        
 8   date         datetime64[ns]
dtypes: datetime64[ns](1), int64(4), object(4)
memory usage: 480.0+ MB

  Total reviews : 6,990,280

── Sample rows ────────────────────────────────
                review_id                 

### Sample dataset

In [8]:
# Sample dataset
df_sample = loader.sample(df, Path(PATHS["RAW_DATA_PATH"]) / "yelp_reviews_sample.parquet")

  Cache hit — loading sample from /content/drive/MyDrive/TEC/8vo IA/Proyecto IA/data/raw/yelp_reviews_sample.parquet


## Data Cleaning

1. **Feature selection**

To reduce memory usage, I will select only the features I'll need (text, sentiment).

2. **Deduplication**

Remove duplicate reviews if any.

3. **Create sentiment column**

As the dataset originally works with stars. I need to create another variable to categorize the dataset in less categories:

- **Negative:** Reviews with 1-2 stars
- **Neutral:** Reviews with 3 stars  
- **Positive:** Reviews with 4-5 stars

This grouping is standard practice in sentiment analysis and reflects how consumers interpret ratings in real-world scenarios.

4. **Dataset balancing**

Yelp reviews are naturally imbalanced (~67% positive, ~23% negative, ~10% neutral). An unbalanced model learns to predict the majority class to maximize accuracy without actually learning sentiment patterns.

Balanced sampling (equal examples per class) forces the model to learn discriminative features for each sentiment rather than exploiting class distribution.

5. **Text cleaning** Normalize and extract text features

In this step I addded some feature for future analysis.

Regarding cleaning, I normalized sequences so that sequences such as “good” and “Good” would be treated as equal. Also, I removed punctuation marks that only add noise and make analysis more difficult.

6. **Tokenization and stop words removal**

Splited the text into individual words and removed irrelevant words.

This is important beaucse ML models can't process text directly; they need numerical inputs. Tokenization is the first step to convert text into numbers.

In [9]:
#@title YelpCleaningPipeline

class YelpCleaningPipeline:
  def __init__(self):
    self.STOP_WORDS  = set(stopwords.words("english"))
    self.OUTPUT_DIR  = Path(PATHS["CLEAN_DATA_PATH"])
    self.OUTPUT_FILE = self.OUTPUT_DIR / "yelp_reviews_clean.parquet"

  def file_exists(self, path: Path = None) -> bool:
    return (path or self.OUTPUT_FILE).exists()

  def save(self, df: pd.DataFrame, path: Path = None) -> Path:
    target = path or self.OUTPUT_FILE
    target.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(target, index=False)
    print(f"  Saved → {target}  ({target.stat().st_size / 1e6:.1f} MB)")
    return target

  def select_features(self, df: pd.DataFrame, features: list) -> pd.DataFrame:
    print(f"  Selecting features: {features}")
    return df[features].copy()

  def drop_duplicates(self, df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates()
    print(f"  Dropped {before - len(df):,} duplicates → {len(df):,} rows remain")
    return df

  def create_sentiment_column(self, df: pd.DataFrame) -> pd.DataFrame:
    print("  Creating sentiment column…")

    def _classify(stars):
      if stars in (1.0, 2.0): return "negative"
      if stars == 3.0:         return "neutral"
      if stars in (4.0, 5.0): return "positive"
      return None

    df = df.copy()
    df["sentiment"] = df["stars"].apply(_classify)

    total = len(df)
    for label, count in df["sentiment"].value_counts().items():
      print(f"    {label:9s}: {count:,}  ({count / total * 100:.1f}%)")

    return df[["text", "sentiment"]]

  def balance_dataset(self, df: pd.DataFrame) -> pd.DataFrame:
    min_count = df["sentiment"].value_counts().min()
    print(f"  Balancing to {min_count:,} reviews per class…")

    df_balanced = (
      df.groupby("sentiment", group_keys=False)
        .apply(lambda g: g.head(min_count))
        .reset_index(drop=True)
    )

    print(f"  Balanced size: {len(df_balanced):,} reviews")
    return df_balanced

  def clean_text(self, df: pd.DataFrame) -> pd.DataFrame:
    print("  Cleaning text…")
    df = df.copy()
    df["text_length"] = df["text"].str.len()
    df["word_count"]  = df["text"].str.split().str.len()
    df["text_clean"]  = df["text"].str.lower().str.replace(r"[^a-zA-Z0-9\s]", "", regex=True)
    return df

  def tokenize_text(self, df: pd.DataFrame) -> pd.DataFrame:
    print("  Tokenizing and removing stop words…")
    df = df.copy()
    df["tokens"] = df["text_clean"].apply(
      lambda x: word_tokenize(x) if isinstance(x, str) else []
    )
    df["tokens_filtered"] = df["tokens"].apply(
      lambda tokens: [w for w in tokens if w not in self.STOP_WORDS and w]
    )
    return df

  def transform(self, df: pd.DataFrame) -> pd.DataFrame:
    """
    Run all cleaning steps and return the processed DataFrame.
    """
    print("Running cleaning pipeline…")
    df = self.select_features(df, ["text", "stars"])
    df = self.drop_duplicates(df)
    df = self.create_sentiment_column(df)
    df = self.balance_dataset(df)
    df = self.clean_text(df)
    df = self.tokenize_text(df)
    print(f"  Pipeline complete → {len(df):,} rows")
    return df

  def load(self, df: pd.DataFrame = None, force: bool = False) -> pd.DataFrame:
    """
    Return the cached parquet if it exists, otherwise run the pipeline and cache it.

      df_clean = YelpCleaningPipeline().load(df_raw)
    """
    if not force and self.file_exists():
      print(f"  Cache hit — loading from {self.OUTPUT_FILE}")
      return pd.read_parquet(self.OUTPUT_FILE)

    if df is None:
      raise ValueError("No cached file found and no DataFrame provided to process.")

    df_clean = self.transform(df)
    self.save(df_clean)
    return df_clean

In [10]:
# Cleaning and feature engineering

pipeline = YelpCleaningPipeline()

df_clean = pipeline.load(df_sample)

  Cache hit — loading from /content/drive/MyDrive/TEC/8vo IA/Proyecto IA/data/clean/yelp_reviews_clean.parquet


## Train, Validation & Test Split


To ensure the model generalizes well to unseen data, I will split the cleaned and balanced dataset into three subsets:

1.  **Training Set (70%)**: Used to train the sentiment analysis models.
2.  **Validation Set (15%)**: Used for hyperparameter tuning and model selection to prevent overfitting.
3.  **Test Set (15%)**: A final hold-out set used to provide an unbiased evaluation of the final model's performance.

I am using a fixed random seed to ensure reproducibility of the splits.

In [11]:
def split_and_save(df: pd.DataFrame, output_path: Path, seed: int = 42) -> None:
  """
  Split: 70% train, 15% validation, 15% test
  """

  df_train, df_temp = train_test_split(df_clean, test_size=0.3, random_state=42)
  df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42)

  print(f"Train: {len(df_train):,}")
  print(f"Validation: {len(df_val):,}")
  print(f"Test: {len(df_test):,}")

  df_train.to_parquet(output_path / "yelp_train.parquet", index=False)
  df_val.to_parquet(output_path / "yelp_val.parquet", index=False)
  df_test.to_parquet(output_path / "yelp_test.parquet", index=False)




In [12]:
split_and_save(df_clean, Path(PATHS["CLEAN_DATA_PATH"]))

Train: 290,514
Validation: 62,253
Test: 62,253
